In [1]:
import re
import pandas as pd
from pathlib import Path


In [2]:
# ===== patterns =====

outer_re = re.compile(r"^=+\s*Outer Iteration\s+(\d+)\s*/\s*(\d+)\s*=+")

# Example:
# [HAM] step   10 | loss=7.945e-04 | grad=3.65e-03 | Δ=-1.50997 h=-0.29001 |
ham_re = re.compile(
    r"\[HAM\]\s*step\s*(\d+)\s*\|"
    r".*?\bloss=([0-9.+-eE]+)\b"
    r".*?\|\s*(?:Δ|delta|Delta)\s*=\s*([0-9.+-eE]+)\b"
    r"\s*h\s*=\s*([0-9.+-eE]+)\b"
)


In [4]:
file_path = Path("./xxz_dmrg_active_403138.out")

rows = []

outer_iter = None
outer_total = None

with file_path.open("r", encoding="utf-8", errors="replace") as f:
    for line in f:

        # outer iteration
        m_outer = outer_re.match(line)
        if m_outer:
            outer_iter = int(m_outer.group(1))
            outer_total = int(m_outer.group(2))
            continue

        # HAM optimization line
        m = ham_re.search(line)
        if m:
            step = int(m.group(1))
            loss = float(m.group(2))
            delta = float(m.group(3))
            h = float(m.group(4))

            rows.append({
                "outer_iter": outer_iter,
                "outer_total": outer_total,
                "step": step,
                "loss": loss,
                "delta": delta,
                "h": h,
            })


In [5]:
df = pd.DataFrame(rows)
df.head()


,outer_iter,outer_total,step,loss,delta,h
0,1,6,1,0.000833,-1.50100,-0.29900
1,1,6,10,0.000794,-1.50997,-0.29001
2,1,6,20,0.000753,-1.51979,-0.28009
3,1,6,30,0.000714,-1.52934,-0.27028
4,1,6,40,0.000678,-1.53856,-0.26061
